# Chest X-Ray Pneumonia Classification

Binary classification of chest X-ray images as **NORMAL** vs **PNEUMONIA**, built as an
end-to-end deep-learning study on medical imaging.

The notebook follows a deliberate learning arc:

1. **Explore** how an image is represented as a tensor.
2. **Augment** the data (domain-aware) and build the DataLoaders.
3. Train a **small CNN from scratch** as a baseline.
4. Apply **transfer learning** (ResNet18, ImageNet) — feature extraction vs. fine-tuning.
5. **Fight overfitting**, rebuild a trustworthy validation set, and evaluate through a
   **medical lens** where *recall* (not accuracy) is the metric that matters most.

> **Environment:** Google Colab + GPU (Tesla T4). The dataset is downloaded via the Kaggle API.


## 0. Environment setup

Image + CNN workloads are far too slow on CPU, so a GPU runtime is required.
In Colab: **Runtime → Change runtime type → Hardware accelerator → GPU**.

In [ ]:
import torch

# Should print True and the GPU name (e.g. Tesla T4).
print("CUDA available:", torch.cuda.is_available())
print("Device name   :", torch.cuda.get_device_name(0))

### 0.1 Download the dataset (Kaggle API)

The dataset is ~2.3 GB, so manual upload is impractical — we pull it directly with the Kaggle API.

1. On Kaggle: **Settings → API → Create New Token** to download `kaggle.json`.
2. Upload `kaggle.json` into the Colab file browser, then run the cell below.

In [ ]:
# Install the Kaggle client and register the API token.
!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download and unzip the "Chest X-Ray Images (Pneumonia)" dataset.
# Tip: use the "Copy API command" button on the Kaggle dataset page if the slug ever changes.
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip

In [ ]:
# The folder name IS the label. Confirm the actual path (it can be nested one level deeper).
!ls chest_xray

## 1. Explore the data — an image is just an array of numbers

To a computer an image is a grid of pixel values, not a picture. The single most important
convention to remember: **PyTorch stores tensors as `[C, H, W]` (channels first)**, while
matplotlib expects `[H, W, C]`. That mismatch is the #1 beginner trap.

`ImageFolder` reads the `NORMAL/` and `PNEUMONIA/` sub-folders and labels the images
automatically from the folder names.

In [ ]:
from torchvision import datasets, transforms

train_dir = "/content/chest_xray/train"

# Minimal transform for exploration: resize to a fixed size, then convert to a tensor.
# ToTensor() does two things: reorders [H,W,C] -> [C,H,W] and rescales pixels 0-255 -> 0.0-1.0.
basic_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(train_dir, transform=basic_tf)

print("classes      :", train_ds.classes)       # ['NORMAL', 'PNEUMONIA']
print("class_to_idx :", train_ds.class_to_idx)   # {'NORMAL': 0, 'PNEUMONIA': 1}
print("num images   :", len(train_ds))           # 5216

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

# Read one batch and inspect its shape: [batch, channels, height, width].
images, labels = next(iter(train_loader))
print("batch images:", images.shape)   # torch.Size([32, 3, 224, 224])
print("batch labels:", labels.shape)   # torch.Size([32])

In [ ]:
import matplotlib.pyplot as plt

# A NORMAL example. Note the .permute(1, 2, 0): [C,H,W] -> [H,W,C] so matplotlib can show it.
img, label = train_ds[0]
print(img.shape, "label:", label, train_ds.classes[label])

plt.imshow(img.permute(1, 2, 0))
plt.title(train_ds.classes[label])
plt.axis("off")
plt.show()

In [ ]:
# A PNEUMONIA example — the lungs tend to look hazier/whiter than a healthy chest.
img, label = train_ds[4000]
print(img.shape, "label:", label, train_ds.classes[label])

plt.imshow(img.permute(1, 2, 0))
plt.title(train_ds.classes[label])
plt.axis("off")
plt.show()

In [ ]:
import numpy as np

# Class imbalance check (the finance-project value_counts instinct, applied to images).
targets = np.array(train_ds.targets)
n_normal    = (targets == 0).sum()
n_pneumonia = (targets == 1).sum()
print("NORMAL   :", n_normal)
print("PNEUMONIA:", n_pneumonia)
print("ratio (pneumonia / normal):", round(n_pneumonia / n_normal, 2))

**Observation.** The training set holds **5,216** images with a class imbalance of about
**3:1** in favour of pneumonia (1,341 normal vs. 3,875 pneumonia). Accuracy alone will be
misleading here — recall on the pneumonia class is what we will track.

There is also a known data-quality issue: the provided `val` folder contains only **16 images**,
far too few to trust. We keep using it for now and rebuild a proper validation set in Section 5.

## 2. Data augmentation & DataLoaders

Augmentation shows the model a slightly different version of each image every epoch
(flip, small rotation, brightness jitter), which discourages memorisation and reduces
overfitting — effectively "free" extra data.

**Domain matters.** Not every augmentation is valid for chest X-rays:

- ❌ **Vertical flip** — an upside-down chest X-ray does not exist in reality.
- ❌ **Large rotations** — X-rays are taken roughly upright; only small angles (±10°) are realistic.
- ⚠️ **Horizontal flip** — debatable (it mirrors the heart), but generally harmless for detecting
  lung opacities; kept here.
- ✅ **Mild brightness jitter** — realistic, since exposure varies between machines.

**Golden rule (same spirit as "fit the scaler on train only"):** augmentation is applied to
**train only**. Validation/test use fixed transforms so evaluation stays honest and reproducible.

In [ ]:
# Two separate pipelines. ToTensor() is the boundary:
# image-space ops (Resize, flips, rotation, jitter) come BEFORE it; Normalize comes AFTER.
# The ImageNet mean/std are used because the ResNet we load in Section 4 expects them.
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Sanity check that augmentation is random: pull the SAME image four times and display it.
# (Normalize is intentionally dropped here — it is for the model, not for human eyes.)
demo_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
])
demo_ds = datasets.ImageFolder(train_dir, transform=demo_tf)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax in axes:
    img, _ = demo_ds[0]                 # re-fetching re-applies the random augmentation
    ax.imshow(img.permute(1, 2, 0))     # [C,H,W] -> [H,W,C]
    ax.axis("off")
plt.suptitle("Same image, four random augmentations")
plt.show()

In [ ]:
# Build train / val / test datasets and loaders (augment train, fixed transforms for eval).
val_dir  = "/content/chest_xray/val"
test_dir = "/content/chest_xray/test"

train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_tf)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)   # shuffle train only
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))
print("train batch:", images.shape)

## 3. Baseline — a small CNN from scratch

Before reaching for transfer learning, we train a small CNN from scratch to *feel* its limits.
Performance is expected to be modest — with only ~5k images a network cannot learn "how to see"
from zero. This weak baseline is exactly what makes the transfer-learning gain visible later.

The architecture stacks **Conv → ReLU → MaxPool** three times (spatial size shrinks, channel
count grows), then **Flatten → Linear**. A dummy tensor is passed through the conv stack to read
off the flatten size instead of computing it by hand.

In [ ]:
import torch.nn as nn

# Pass a dummy image through the full stack to confirm the output shape / flatten size.
dummy = torch.randn(1, 3, 224, 224)
model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),   # 224 -> 112
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 112 -> 56
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 56  -> 28
    nn.Flatten(),
    nn.Linear(64 * 28 * 28, 2),      # flatten size = 64 * 28 * 28 = 50176
)
print("output shape:", model(dummy).shape)   # torch.Size([1, 2])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)                       # model to GPU

loss_fn   = nn.CrossEntropyLoss()              # 2-class output -> integer labels work directly
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# The same four-line training loop as the tabular project, plus .to(device).
n_epochs = 3
for epoch in range(n_epochs):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)   # data to GPU
        optimizer.zero_grad()                                   # (1) reset grads
        outputs = model(images)                                 # (2) forward
        loss = loss_fn(outputs, labels)                         # (3) how wrong?
        loss.backward()                                         # (4) backward
        optimizer.step()                                        # (5) update
        running += loss.item()
    train_loss = running / len(train_loader)

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_running += loss_fn(model(images), labels).item()
    val_loss = val_running / len(val_loader)

    print(f"epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Evaluate on the held-out test folder, collecting predictions batch by batch.
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()   # .cpu() before numpy(); argmax = class
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# recall here = "of all real pneumonia cases, how many did we catch?" (pos_label = 1 = PNEUMONIA)
print("recall   :", recall_score(all_labels, all_preds))
print("precision:", precision_score(all_labels, all_preds))
print("F1       :", f1_score(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))

**Baseline result (from scratch).** Recall ≈ 0.99, precision ≈ 0.69, F1 ≈ 0.82.
The model catches almost every pneumonia case but raises many false alarms — a weak-but-working
starting point. Now we see how far transfer learning pushes this.

## 4. Transfer learning — ResNet18 (the heart of the project)

Idea: borrow the "seeing" ability of a network already trained on ImageNet (1.4M images).
The **early layers** (edges, textures) are universal and reused as-is; only the **final classifier**
is swapped for our 2-class problem and trained.

We use `torchvision` (not HuggingFace) here because the freeze/replace mechanics are transparent
and easy to reason about.

### 4.1 Feature extraction (freeze)

Freeze the whole backbone, replace `fc` with a fresh 2-class layer, and train **only** that layer.
Fast, and low overfitting risk when data is scarce.

In [ ]:
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)   # ImageNet weights

for param in model.parameters():
    param.requires_grad = False            # freeze everything...

num_features = model.fc.in_features        # ResNet18 -> 512 (read it, don't hardcode)
model.fc = nn.Linear(num_features, 2)      # ...then replace fc (new layer trains by default)

model = model.to(device)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)   # optimise fc only
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# Same training loop; only the model changed. Frozen backbone -> each epoch is fast.
n_epochs = 3
for epoch in range(n_epochs):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_loss = running / len(train_loader)

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_running += loss_fn(model(images), labels).item()
    val_loss = val_running / len(val_loader)
    print(f"epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("recall   :", recall_score(all_labels, all_preds))
print("precision:", precision_score(all_labels, all_preds))
print("F1       :", f1_score(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))

### 4.2 Fine-tuning

Unfreeze the network and continue training with a **much smaller** learning rate (1e-4). The
borrowed weights are already excellent, so large steps would destroy them — we nudge gently.

In [ ]:
for param in model.parameters():
    param.requires_grad = True             # unfreeze all

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)   # small LR is essential

n_epochs = 3
for epoch in range(n_epochs):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_loss = running / len(train_loader)

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_running += loss_fn(model(images), labels).item()
    val_loss = val_running / len(val_loader)
    print(f"epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("recall   :", recall_score(all_labels, all_preds))
print("precision:", precision_score(all_labels, all_preds))
print("F1       :", f1_score(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))

**First three-model comparison** (all on the same 624-image test set, 3 epochs each):

| Model | Recall | Precision | F1 | Missed pneumonia (FN) | False alarms (FP) |
|---|---|---|---|---|---|
| Small CNN (scratch)        | 0.99 | 0.69 | 0.82 | 3 | 172 |
| ResNet18 (freeze)          | 0.98 | 0.85 | **0.91** | 8 | 65 |
| ResNet18 (fine-tune)       | **1.00** | 0.75 | 0.86 | 1 | 128 |

Transfer learning clearly beats training from scratch. Notably, the **frozen** feature extractor
already gives the best precision/F1 balance, while fine-tuning maximises recall (misses only 1 of
390 pneumonia cases). Section 5 refines this with a trustworthy validation set and overfitting
control.

## 5. Overfitting control, a proper validation set, and medical evaluation

Three goals: (1) diagnose overfitting with loss curves and fight it, (2) replace the unreliable
16-image validation set, and (3) evaluate the way a clinician would — where a **missed pneumonia
(false negative)** is the most expensive mistake.

### 5.1 Rebuild a stratified validation set from the training data

The stock `val` folder has only 16 images, so we carve a 20% validation split out of `train`
instead (the original `test` folder is untouched for final evaluation).

Trick: train needs the **augmented** transform while validation needs the **fixed** transform,
but both come from the same folder. Solution — build **two** `ImageFolder`s over the same
directory and apply the **same index split** to each, so the subsets never overlap yet carry
different transforms. `stratify` keeps the 3:1 class ratio in both splits.

In [ ]:
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split

train_ds_aug  = datasets.ImageFolder(train_dir, transform=train_tf)   # augmented (for training)
train_ds_eval = datasets.ImageFolder(train_dir, transform=eval_tf)    # fixed     (for validation)

indices = list(range(len(train_ds_aug)))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, stratify=train_ds_aug.targets, random_state=42
)

train_subset = Subset(train_ds_aug,  train_idx)   # train indices from the augmented dataset
val_subset   = Subset(train_ds_eval, val_idx)     # val indices   from the fixed dataset

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_subset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds,      batch_size=64, shuffle=False)   # original test folder

print("train:", len(train_subset), " val:", len(val_subset), " test:", len(test_ds))

### 5.2 Robust transfer pipeline — dropout + early stopping + best-model checkpoint

We add three defences against overfitting on top of the augmentation already in place:

- **Dropout** in the classifier head (only active in `train()` mode).
- **Early stopping** — stop when validation loss stops improving (patience = 3).
- **Best-model checkpoint** — save `state_dict` at the lowest validation loss and reload it,
  because the final epoch is not always the best.

**Stage 1 — feature extraction with the new validation set.**

In [ ]:
# Stage 1: freeze backbone, dropout head, train fc only, with early stopping + checkpoint.
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, 2),
)
model = model.to(device)

loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

best_val = float("inf")
patience, wait = 3, 0
train_losses, val_losses = [], []

n_epochs = 6
for epoch in range(n_epochs):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_loss = running / len(train_loader)

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_running += loss_fn(model(images), labels).item()
    val_loss = val_running / len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_model.pth")   # checkpoint the best moment
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Stage 1 (feature extraction) loss curves")
plt.show()

print(model.load_state_dict(torch.load("best_model.pth")))   # reload best weights

**Stage 2 — fine-tuning.** Reload the best feature-extraction weights, unfreeze the whole
network, and fine-tune with a small learning rate, keeping the same early-stopping/checkpoint
machinery.

In [ ]:
# Stage 2: unfreeze all and fine-tune from the best Stage-1 checkpoint.
model.load_state_dict(torch.load("best_model.pth"))

for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)   # small LR for fine-tuning

best_val = float("inf")
patience, wait = 3, 0
train_losses, val_losses = [], []

n_epochs = 6
for epoch in range(n_epochs):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_loss = running / len(train_loader)

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_running += loss_fn(model(images), labels).item()
    val_loss = val_running / len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_finetuned.pth")
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Stage 2 (fine-tuning) loss curves")
plt.show()

print(model.load_state_dict(torch.load("best_finetuned.pth")))

In [ ]:
# Final test-set metrics for the fine-tuned model.
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("recall   :", recall_score(all_labels, all_preds))
print("precision:", precision_score(all_labels, all_preds))
print("F1       :", f1_score(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))

### 5.3 Reading the confusion matrix in medical language

With labels `NORMAL = 0`, `PNEUMONIA = 1`:

- **TP** — pneumonia correctly flagged. ✅
- **FN (missed pneumonia) — the worst error.** A sick patient sent home untreated; the illness
  can worsen. This is the most expensive mistake in the clinic.
- **FP (false alarm)** — a healthy patient flagged as pneumonia. Causes extra tests and anxiety,
  but it is the *safe* kind of error.
- **TN** — normal correctly cleared. ✅

Conclusion: **minimising FN = maximising recall** is the clinically correct objective. That is
exactly why recall — not accuracy — sits at the centre of this project. The fine-tuned model
misses essentially no pneumonia (recall ≈ 1.00); its lower precision means more false alarms,
which is the acceptable trade-off in this setting.

### 5.4 Error analysis — look at the missed pneumonia (FN) cases

Numbers are not enough — we inspect the X-rays the model got wrong, focusing on false negatives.
The normalisation is reversed (`img * std + mean`) so the images look natural to the eye.

In [ ]:
# Collect misclassified cases, then keep only false negatives (true=PNEUMONIA, pred=NORMAL).
wrong = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(1).cpu()
        for i in range(len(labels)):
            if preds[i] != labels[i]:
                wrong.append((images[i], labels[i].item(), preds[i].item()))

fn_cases = [w for w in wrong if w[1] == 1 and w[2] == 0]   # missed pneumonia
print("total misclassified:", len(wrong), " | missed pneumonia (FN):", len(fn_cases))

# Undo ImageNet normalisation for display.
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (img, true, pred) in zip(axes, fn_cases[:4]):
    ax.imshow((img * std + mean).permute(1, 2, 0).clamp(0, 1))
    ax.set_title(f"Actual: {true}  Pred: {pred}")
    ax.axis("off")
plt.suptitle("Missed pneumonia (false negatives)")
plt.show()

### 5.5 Baseline re-run under the same harness (fair comparison)

For an apples-to-apples comparison, the from-scratch CNN is re-trained with the same rebuilt
validation set, dropout, and early stopping.

In [ ]:
# Small CNN + dropout, trained with the carved validation set and early stopping.
model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Dropout(0.5),
    nn.Linear(64 * 28 * 28, 2),
)
model = model.to(device)

loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val = float("inf")
patience, wait = 3, 0
train_losses, val_losses = [], []

n_epochs = 6
for epoch in range(n_epochs):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_loss = running / len(train_loader)

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_running += loss_fn(model(images), labels).item()
    val_loss = val_running / len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_basic_cnn.pth")
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Small CNN loss curves")
plt.show()
print(model.load_state_dict(torch.load("best_basic_cnn.pth")))

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("recall   :", recall_score(all_labels, all_preds))
print("precision:", precision_score(all_labels, all_preds))
print("F1       :", f1_score(all_labels, all_preds))

## 6. Results & conclusion

**Final comparison** (held-out test set, 624 images: 234 normal + 390 pneumonia):

| Model | Recall (pneumonia) | Precision | F1 |
|---|---|---|---|
| Small CNN (from scratch) | 0.99 | 0.72 | 0.83 |
| ResNet18 — feature extraction (freeze) | 0.98 | 0.85 | **0.91** |
| ResNet18 — fine-tuning | **1.00** | 0.76 | 0.86 |

*Numbers are from Colab (Tesla T4) runs; augmentation/dropout are stochastic, so re-runs vary
slightly. The freeze row reflects the Section-4 run; the scratch and fine-tune rows reflect the
Section-5 refined pipeline. All are evaluated on the same held-out test folder.*

**Takeaways**

- **Transfer learning is the workhorse.** With only ~5k images, borrowing ImageNet features beats
  training from scratch on precision and F1, at a fraction of the effort.
- **Recall is the right target for this domain.** Every model reaches ≈0.98–1.00 recall; the
  fine-tuned network misses essentially no pneumonia. Its lower precision (more false alarms) is
  the safe trade-off medically — a false alarm costs a follow-up, a missed case can cost a life.
- **Freeze vs. fine-tune is a real choice.** The frozen feature extractor gives the best overall
  balance (F1 ≈ 0.91); fine-tuning maximises recall. Which to ship depends on whether the clinical
  goal is "miss nothing" or "balanced triage".
- **Honest validation matters.** Replacing the 16-image validation folder with a stratified split
  carved from train, plus dropout and early stopping, makes the reported numbers trustworthy.

**Reused foundations from the earlier tabular project:** the imbalance instinct, confusion-matrix /
recall-first evaluation, the four-line training loop, stratified splitting, and the "only touch the
training data" principle — all carried over directly to images.
